# HI 5312 – Introduction to Data Analysis
## Module 4: Cleaning and Exploring Healthcare Data with Python

**Estimated time:** 6–8 hours  
**Environment:** GitHub Codespaces  
**Kernel:** Python 3.12.12  
**Database:** PostgreSQL (`hi5304`)  
**Primary tools:** Python, pandas, SQLAlchemy, PostgreSQL, and matplotlib

> **Course progression:** Better Questions → Better Data → Better Analysis → Better Decisions → Better Health

## Clinical Request

In Module 3, you used SQL to retrieve records from the course PostgreSQL database. The Director of Quality Improvement now wants you to prepare those records for analysis.

Your request is:

> **Create a reproducible Python workflow that imports healthcare data, evaluates its quality, cleans the data, creates useful analytic variables, summarizes the population, and identifies patterns that may require further investigation.**

Your work must be transparent enough that another analyst can understand and reproduce every step.

## Why This Matters

SQL retrieves data. Python helps analysts transform those records into an analysis-ready dataset.

Healthcare data commonly contain missing values, duplicate records, inconsistent categories, incorrect data types, implausible measurements, and variables that must be derived before analysis. Cleaning these issues manually in a spreadsheet can be difficult to reproduce. A Python notebook records the complete sequence of decisions, code, results, and interpretation.

The goal is not to make data look perfect. The goal is to understand the data, document limitations, and make defensible decisions.

## Learning Objectives

After completing this module, you should be able to:

1. Confirm and use the Python 3.12.12 kernel.
2. Explain variables, data types, lists, dictionaries, conditions, loops, and functions.
3. Import Python libraries and interpret common error messages.
4. Connect Python to the course PostgreSQL database.
5. load SQL results into a pandas DataFrame.
6. Inspect rows, columns, dimensions, data types, and descriptive summaries.
7. Select, filter, sort, and rename variables.
8. Identify missing values, duplicate records, inconsistent categories, and implausible values.
9. Convert data types and create derived variables.
10. Summarize data with `groupby`, frequency tables, and cross-tabulations.
11. Create basic exploratory visualizations with matplotlib.
12. Export an analysis-ready dataset and document cleaning decisions.

## Essential Question

> **How can Python help me transform raw healthcare data into trustworthy, analysis-ready information?**

## Professional Competency

**Develop a reproducible Python workflow that evaluates, cleans, summarizes, and documents healthcare data.**

# Part 1 – Begin with the Correct Python Kernel

At the upper right of the notebook, confirm that the selected kernel is:

> **Python 3.12.12**

A kernel is the computational environment that executes notebook code. Using the course kernel helps ensure that students have the same Python version and packages.

In [1]:
import sys
import platform

print("Python version:", sys.version)
print("Platform:", platform.platform())

assert sys.version_info[:2] == (3, 12), (
    "Select the Python 3.12.12 kernel before continuing."
)

Python version: 3.12.13 (main, Jun 24 2026, 04:58:38) [GCC 12.2.0]
Platform: Linux-6.8.0-1064-azure-x86_64-with-glibc2.36


## Python as a Language

Python instructions are executed in order. A variable stores a value under a meaningful name.

Common Python data types include:

| Type | Example | Typical use |
|---|---|---|
| Integer | `65` | counts, whole-number values |
| Float | `27.4` | measurements and decimal values |
| String | `"Female"` | categories and text |
| Boolean | `True` | yes/no conditions |
| List | `[120, 130, 142]` | ordered collection |
| Dictionary | `{"sbp": 142}` | named key-value collection |

In [2]:
patient_id = 1001
age_years = 67
bmi = 29.4
sex = "Female"
has_diabetes = True
blood_pressures = [138, 142, 136]
patient = {"patient_id": patient_id, "age": age_years, "bmi": bmi}

print(type(patient_id))
print(type(bmi))
print(type(sex))
print(type(has_diabetes))
print(patient)

<class 'int'>
<class 'float'>
<class 'str'>
<class 'bool'>
{'patient_id': 1001, 'age': 67, 'bmi': 29.4}


## Conditions and Functions

A condition allows code to make a decision. A function packages reusable logic.

In [3]:
def classify_systolic_bp(sbp):
    """Return a simple instructional category for systolic blood pressure."""
    if sbp < 120:
        return "Below 120"
    elif sbp < 140:
        return "120–139"
    return "140 or higher"

for value in blood_pressures:
    print(value, classify_systolic_bp(value))

138 120–139
142 140 or higher
136 120–139


## Guided Laboratory 1 – Python Fundamentals

Complete the following tasks:

1. Create variables for a patient identifier, age, BMI, and smoking status.
2. Store three systolic blood pressure measurements in a list.
3. Calculate their mean using `sum()` and `len()`.
4. Write a function that returns `"Review"` when the mean is 140 or higher and `"Routine"` otherwise.

In [ ]:
# Guided Laboratory 1 workspace

# patient_id =
# age =
# bmi =
# smoking_status =
# systolic_readings = [...] 

# mean_sbp = sum(systolic_readings) / len(systolic_readings)
# print(mean_sbp)

# Part 2 – Import the Analysis Libraries

A library is a collection of reusable code. This module uses:

- **pandas** for tabular data;
- **SQLAlchemy** for the PostgreSQL connection;
- **matplotlib** for visualizations;
- **NumPy** for numeric operations.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, inspect, text

print("pandas:", pd.__version__)
print("NumPy:", np.__version__)

# Part 3 – Connect to PostgreSQL from Python

The course database is already running inside the Codespace.

| Setting | Value |
|---|---|
| Host | `db` |
| Port | `5432` |
| Database | `hi5304` |
| Username | `hi5304_user` |
| Password | `hi5304` |

The code below uses environment variables when available and otherwise uses the instructional credentials. In production systems, credentials should be stored as secure secrets rather than written into source code.

In [ ]:
DB_HOST = os.getenv("DB_HOST", "db")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "hi5304")
DB_USER = os.getenv("DB_USER", "hi5304_user")
DB_PASSWORD = os.getenv("DB_PASSWORD", "hi5304")

connection_url = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    database_name = connection.execute(text("SELECT current_database();")).scalar()

print("Connected to:", database_name)

## Inspect the Database Before Selecting a Table

Do not assume a table or variable exists. Inspect the available schema first.

In [ ]:
inspector = inspect(engine)
tables = sorted(inspector.get_table_names(schema="public"))

print("Available public tables:")
for table in tables:
    print("-", table)

In [ ]:
def describe_table(table_name):
    """Return the column names and SQL data types for a public table."""
    columns = inspector.get_columns(table_name, schema="public")
    return pd.DataFrame(
        {
            "column_name": [column["name"] for column in columns],
            "data_type": [str(column["type"]) for column in columns],
            "nullable": [column.get("nullable") for column in columns],
        }
    )

for candidate in ["cardio", "patients", "bp_readings", "medications"]:
    if candidate in tables:
        print()
        print(candidate)
        display(describe_table(candidate))

## Select a Primary Table

The notebook prefers the `cardio` table because it typically contains a broad set of demographic, biometric, behavioral, and outcome variables. If `cardio` is unavailable, the code uses `patients`, followed by the first available table.

In [ ]:
preferred_tables = ["cardio", "patients", "cardio1", "cardio10"]
primary_table = next((name for name in preferred_tables if name in tables), None)

if primary_table is None:
    if not tables:
        raise RuntimeError("No public tables were found in the database.")
    primary_table = tables[0]

print("Primary table selected:", primary_table)
display(describe_table(primary_table))

## Load SQL Results into pandas

`pd.read_sql()` executes a SQL query and returns the result as a DataFrame. A DataFrame is a two-dimensional table with labeled rows and columns.

In [ ]:
query = f'SELECT * FROM public."{primary_table}";'
raw_df = pd.read_sql(query, engine)

print("Rows and columns:", raw_df.shape)
raw_df.head()

# Part 4 – Understand the DataFrame

Before changing the data, inspect its structure.

Useful methods and attributes include:

- `df.head()` and `df.tail()`
- `df.shape`
- `df.columns`
- `df.info()`
- `df.dtypes`
- `df.describe()`
- `df.nunique()`

In [ ]:
print("Shape:", raw_df.shape)
print()
print("Columns:")
print(raw_df.columns.tolist())
print()
print("Data types:")
print(raw_df.dtypes)

In [ ]:
raw_df.info()

In [ ]:
raw_df.describe(include="all").transpose()

## Create a Working Copy

Preserve the raw import. Perform cleaning on a copy so that the original remains available for comparison.

In [ ]:
df = raw_df.copy()
print("Raw copy preserved as raw_df")
print("Working copy created as df")

# Part 5 – Select, Filter, Sort, and Rename

Pandas provides several ways to work with rows and columns.

```python
# One column
df["column_name"]

# Several columns
df[["column_a", "column_b"]]

# Filter rows
df[df["numeric_column"] >= 100]

# Sort rows
df.sort_values("numeric_column", ascending=False)
```

Because healthcare databases use different schemas, first select columns that actually exist.

In [ ]:
# Display up to six available columns.
selected_columns = df.columns[: min(6, len(df.columns))].tolist()
print("Selected columns:", selected_columns)
df[selected_columns].head()

In [ ]:
# Identify numeric columns that can be used for sorting and filtering.
numeric_columns = df.select_dtypes(include="number").columns.tolist()
print("Numeric columns:", numeric_columns)

if numeric_columns:
    example_numeric = numeric_columns[0]
    display(df.sort_values(example_numeric, ascending=False).head())
else:
    print("No numeric columns were identified.")

## Guided Laboratory 2 – DataFrame Navigation

Using the working DataFrame:

1. Display the last five rows.
2. Report the number of rows and columns.
3. Create a list of all column names.
4. Select three columns.
5. Sort by one numeric column.
6. Filter the DataFrame using a clinically meaningful condition.
7. Explain what one row represents.

In [ ]:
# Guided Laboratory 2 workspace

# df.tail()
# df.shape
# df.columns.tolist()
# df[[...]]
# df.sort_values(...)
# df[df[...] ...]

# Part 6 – Evaluate Data Quality with pandas

A reproducible data-quality review should assess:

- missing values;
- duplicate rows or identifiers;
- data types;
- category consistency;
- valid ranges;
- logical consistency;
- fitness for purpose.

## Missing Values

`isna()` identifies missing values. Summing the Boolean results counts missing observations.

In [ ]:
missing_summary = pd.DataFrame(
    {
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(1),
    }
).sort_values("missing_percent", ascending=False)

missing_summary

## Duplicate Records

A full-row duplicate repeats every value. A duplicate identifier is more specific and depends on the unit of analysis.

In [ ]:
full_row_duplicates = int(df.duplicated().sum())
print("Full-row duplicates:", full_row_duplicates)

identifier_candidates = [
    column for column in df.columns
    if column.lower() in {"id", "patient_id", "person_id", "subject_id"}
]

if identifier_candidates:
    identifier = identifier_candidates[0]
    duplicate_identifiers = int(df.duplicated(subset=[identifier]).sum())
    print(f"Duplicate values in {identifier}:", duplicate_identifiers)
else:
    identifier = None
    print("No common identifier column was automatically identified.")

## Category Consistency

Text categories may differ because of capitalization, spacing, abbreviations, or spelling. Review the unique values before recoding.

In [ ]:
text_columns = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()
print("Text or categorical columns:", text_columns)

for column in text_columns[:5]:
    print()
    print(column)
    print(df[column].value_counts(dropna=False).head(15))

## Numeric Range Review

A minimum or maximum does not automatically prove that a value is incorrect. It identifies observations that should be evaluated using variable definitions, units, clinical context, and source documentation.

In [ ]:
if numeric_columns:
    range_review = df[numeric_columns].agg(["count", "min", "median", "mean", "max"]).transpose()
    display(range_review)
else:
    print("No numeric columns are available for range review.")

## A Reusable Data-Quality Function

Functions reduce repetition and make workflows easier to reproduce.

In [ ]:
def data_quality_report(dataframe):
    """Create a column-level data-quality summary."""
    report = pd.DataFrame(index=dataframe.columns)
    report["data_type"] = dataframe.dtypes.astype(str)
    report["non_missing"] = dataframe.notna().sum()
    report["missing"] = dataframe.isna().sum()
    report["missing_percent"] = (dataframe.isna().mean() * 100).round(1)
    report["unique_values"] = dataframe.nunique(dropna=True)
    return report.sort_values(["missing_percent", "unique_values"], ascending=[False, True])

quality_report = data_quality_report(df)
quality_report

# Part 7 – Clean the Data

Cleaning decisions should be explicit and documented. Common actions include:

- standardizing column names;
- converting data types;
- trimming text;
- standardizing categories;
- removing exact duplicates;
- retaining, excluding, or imputing missing values;
- flagging implausible values rather than silently deleting them.

## Standardize Column Names

The following transformation converts names to lowercase and replaces spaces or punctuation with underscores.

In [ ]:
import re

original_columns = df.columns.tolist()

def clean_column_name(name):
    cleaned = re.sub(r"[^0-9a-zA-Z]+", "_", str(name).strip().lower())
    return cleaned.strip("_")

df.columns = [clean_column_name(column) for column in df.columns]

column_name_changes = pd.DataFrame(
    {"original": original_columns, "cleaned": df.columns.tolist()}
)
column_name_changes

## Trim Text Values

Do not automatically force every category to uppercase or lowercase without understanding its meaning. Trimming leading and trailing spaces is usually a safe first step.

In [ ]:
text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()

for column in text_columns:
    df[column] = df[column].map(lambda value: value.strip() if isinstance(value, str) else value)

print("Trimmed text columns:", text_columns)

## Convert Data Types Carefully

Use `pd.to_numeric()` and `pd.to_datetime()` when a column is stored as the wrong type. The `errors="coerce"` option converts unparseable values to missing values, which must then be reviewed.

In [ ]:
# Identify columns whose names suggest dates.
date_candidates = [
    column for column in df.columns
    if any(token in column for token in ["date", "time", "dob", "birth"])
]

print("Potential date columns:", date_candidates)

# Example conversion pattern (uncomment after reviewing a column):
# df["visit_date"] = pd.to_datetime(df["visit_date"], errors="coerce")

## Remove Exact Duplicate Rows

Removing exact duplicates may be reasonable when each row should be unique. Duplicate patient identifiers are not automatically errors because one patient may have multiple encounters, medications, or measurements.

In [ ]:
rows_before = len(df)
df = df.drop_duplicates().copy()
rows_after = len(df)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Exact duplicate rows removed:", rows_before - rows_after)

# Part 8 – Create Analytic Variables

Derived variables should be created from documented definitions and units. The helper code below creates common variables only when the required source columns exist.

In [ ]:
def first_existing(columns, candidates):
    """Return the first candidate column found, or None."""
    return next((candidate for candidate in candidates if candidate in columns), None)

age_column = first_existing(df.columns, ["age", "age_years", "age_days"])
height_column = first_existing(df.columns, ["height", "height_cm"])
weight_column = first_existing(df.columns, ["weight", "weight_kg"])
sbp_column = first_existing(df.columns, ["ap_hi", "sbp", "systolic_bp", "systolic"])
dbp_column = first_existing(df.columns, ["ap_lo", "dbp", "diastolic_bp", "diastolic"])

print("Age column:", age_column)
print("Height column:", height_column)
print("Weight column:", weight_column)
print("Systolic column:", sbp_column)
print("Diastolic column:", dbp_column)

In [ ]:
# Convert age in days to years when that is the apparent source variable.
if age_column == "age_days":
    df["age_years"] = df[age_column] / 365.25
elif age_column == "age" and pd.api.types.is_numeric_dtype(df[age_column]):
    # The common cardio dataset stores age in days. Values above 150 are treated as days.
    median_age_value = df[age_column].median()
    if pd.notna(median_age_value) and median_age_value > 150:
        df["age_years"] = df[age_column] / 365.25
    else:
        df["age_years"] = df[age_column]
elif age_column is not None:
    df["age_years"] = pd.to_numeric(df[age_column], errors="coerce")

if height_column and weight_column:
    height_m = pd.to_numeric(df[height_column], errors="coerce") / 100
    weight_kg = pd.to_numeric(df[weight_column], errors="coerce")
    df["bmi_calculated"] = weight_kg / (height_m ** 2)

if sbp_column and dbp_column:
    sbp = pd.to_numeric(df[sbp_column], errors="coerce")
    dbp = pd.to_numeric(df[dbp_column], errors="coerce")
    df["pulse_pressure"] = sbp - dbp

created = [column for column in ["age_years", "bmi_calculated", "pulse_pressure"] if column in df]
print("Derived variables created:", created)

## Create Review Flags

A review flag preserves the original observation while identifying values that may require verification. Thresholds below are instructional examples and should be adapted to the variable definitions and clinical context.

In [ ]:
if sbp_column:
    sbp = pd.to_numeric(df[sbp_column], errors="coerce")
    df["sbp_review_flag"] = ~sbp.between(60, 260) & sbp.notna()

if dbp_column:
    dbp = pd.to_numeric(df[dbp_column], errors="coerce")
    df["dbp_review_flag"] = ~dbp.between(30, 160) & dbp.notna()

if "bmi_calculated" in df:
    df["bmi_review_flag"] = ~df["bmi_calculated"].between(10, 80) & df["bmi_calculated"].notna()

flag_columns = [column for column in df.columns if column.endswith("_review_flag")]

if flag_columns:
    display(df[flag_columns].sum().rename("flagged_records").to_frame())
else:
    print("No review flags were created because the required variables were unavailable.")

# Part 9 – Exploratory Data Analysis

Exploratory data analysis describes the dataset before formal statistical testing. It helps identify distributions, groups, relationships, missingness, and unusual observations.

## Frequency Tables

In [ ]:
# Select likely categorical columns: text fields or numeric fields with few unique values.
categorical_candidates = []
for column in df.columns:
    unique_count = df[column].nunique(dropna=True)
    if df[column].dtype == "object" or unique_count <= 10:
        categorical_candidates.append(column)

print("Potential categorical columns:", categorical_candidates[:20])

if categorical_candidates:
    example_category = categorical_candidates[0]
    display(df[example_category].value_counts(dropna=False).rename("count").to_frame())

## Grouped Summaries

`groupby()` divides the DataFrame into groups, applies a summary function, and combines the results.

In [ ]:
clean_numeric_columns = df.select_dtypes(include="number").columns.tolist()
clean_categorical_candidates = [
    column for column in categorical_candidates
    if column in df.columns and df[column].nunique(dropna=True) >= 2
]

if clean_categorical_candidates and clean_numeric_columns:
    group_column = clean_categorical_candidates[0]
    value_column = next(
        (column for column in ["age_years", "bmi_calculated", sbp_column, dbp_column]
         if column and column in clean_numeric_columns),
        clean_numeric_columns[0],
    )

    grouped_summary = (
        df.groupby(group_column, dropna=False)[value_column]
        .agg(patient_count="count", mean="mean", median="median", minimum="min", maximum="max")
        .round(1)
        .reset_index()
    )

    print(f"Grouped by {group_column}; summarized {value_column}")
    display(grouped_summary)
else:
    print("The table does not contain a suitable categorical and numeric pair.")

## Cross-Tabulations

A cross-tabulation shows the joint frequency of two categorical variables.

In [ ]:
if len(clean_categorical_candidates) >= 2:
    row_variable = clean_categorical_candidates[0]
    column_variable = clean_categorical_candidates[1]
    crosstab = pd.crosstab(
        df[row_variable],
        df[column_variable],
        margins=True,
        dropna=False,
    )
    print(f"Rows: {row_variable}; columns: {column_variable}")
    display(crosstab)
else:
    print("At least two categorical variables are required for a cross-tabulation.")

# Part 10 – Visual Exploration with matplotlib

A useful chart should answer a question. Always provide a title, axis labels, and a brief interpretation.

## Histogram

In [ ]:
plot_numeric = next(
    (column for column in ["age_years", "bmi_calculated", sbp_column, dbp_column]
     if column and column in df.columns),
    clean_numeric_columns[0] if clean_numeric_columns else None,
)

if plot_numeric:
    values = pd.to_numeric(df[plot_numeric], errors="coerce").dropna()
    plt.figure(figsize=(8, 5))
    plt.hist(values, bins=20, edgecolor="black")
    plt.title(f"Distribution of {plot_numeric}")
    plt.xlabel(plot_numeric)
    plt.ylabel("Number of records")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric variable is available for a histogram.")

## Bar Chart

In [ ]:
if clean_categorical_candidates:
    plot_category = clean_categorical_candidates[0]
    counts = df[plot_category].value_counts(dropna=False).head(10)

    plt.figure(figsize=(8, 5))
    counts.plot(kind="bar")
    plt.title(f"Records by {plot_category}")
    plt.xlabel(plot_category)
    plt.ylabel("Number of records")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No categorical variable is available for a bar chart.")

## Boxplot

In [ ]:
if plot_numeric:
    values = pd.to_numeric(df[plot_numeric], errors="coerce").dropna()
    plt.figure(figsize=(8, 4))
    plt.boxplot(values, vert=False)
    plt.title(f"Boxplot of {plot_numeric}")
    plt.xlabel(plot_numeric)
    plt.tight_layout()
    plt.show()

## Scatterplot

In [ ]:
preferred_pair = None
for first, second in [
    ("age_years", sbp_column),
    ("bmi_calculated", sbp_column),
    (height_column, weight_column),
]:
    if first and second and first in df.columns and second in df.columns:
        preferred_pair = (first, second)
        break

if preferred_pair is None and len(clean_numeric_columns) >= 2:
    preferred_pair = (clean_numeric_columns[0], clean_numeric_columns[1])

if preferred_pair:
    x_column, y_column = preferred_pair
    plot_df = df[[x_column, y_column]].apply(pd.to_numeric, errors="coerce").dropna()

    plt.figure(figsize=(8, 5))
    plt.scatter(plot_df[x_column], plot_df[y_column], alpha=0.5)
    plt.title(f"{y_column} by {x_column}")
    plt.xlabel(x_column)
    plt.ylabel(y_column)
    plt.tight_layout()
    plt.show()
else:
    print("Two numeric variables are required for a scatterplot.")

# Part 11 – Document the Cleaning Process

A cleaning log creates an audit trail. It should record what was assessed, what was changed, why it was changed, and how many records were affected.

In [ ]:
cleaning_log = pd.DataFrame(
    [
        {
            "step": 1,
            "action": "Imported data from PostgreSQL",
            "reason": "Create reproducible source extract",
            "records_affected": len(raw_df),
        },
        {
            "step": 2,
            "action": "Standardized column names",
            "reason": "Create consistent Python-friendly names",
            "records_affected": len(df.columns),
        },
        {
            "step": 3,
            "action": "Trimmed text values",
            "reason": "Remove leading and trailing whitespace",
            "records_affected": "Review by column",
        },
        {
            "step": 4,
            "action": "Removed exact duplicate rows",
            "reason": "Prevent repeated identical observations",
            "records_affected": rows_before - rows_after,
        },
        {
            "step": 5,
            "action": "Created derived variables and review flags",
            "reason": "Support analysis while preserving source values",
            "records_affected": "See derived and flag columns",
        },
    ]
)

cleaning_log

## Export Analysis-Ready Files

CSV is broadly portable. Parquet preserves data types more effectively and is often more efficient. The course export below creates a CSV file in the notebook directory.

In [ ]:
output_file = "module4_analysis_ready.csv"
df.to_csv(output_file, index=False)
print(f"Saved {len(df):,} rows and {len(df.columns):,} columns to {output_file}")

# Guided Laboratory 3 – Complete Data-Cleaning Workflow

Using a table selected from the PostgreSQL database:

1. Import the complete table into pandas.
2. Preserve a raw copy.
3. Inspect its dimensions, columns, data types, and summary statistics.
4. Calculate missing counts and percentages.
5. Identify full-row duplicates.
6. Identify the likely record identifier.
7. Review the minimum and maximum of every numeric variable.
8. Review the values of at least three categorical variables.
9. Standardize column names.
10. Create at least one derived variable or review flag.
11. Produce a grouped summary.
12. Create one useful chart.
13. Export the analysis-ready file.
14. Document every cleaning decision.

In [ ]:
# Guided Laboratory 3 workspace

# 1. Select and import a table
# 2. Preserve a raw copy
# 3. Inspect structure
# 4. Assess missingness
# 5. Review duplicates
# 6. Identify key
# 7. Review numeric ranges
# 8. Review categories
# 9. Standardize names
# 10. Create variable or flag
# 11. Grouped summary
# 12. Visualization
# 13. Export
# 14. Cleaning log

# Practice Laboratory – Python and pandas

Complete the following exercises in new code cells. Use the course database and the DataFrame created in this notebook.

1. Display the first 10 rows.
2. Display the final 10 rows.
3. Report the DataFrame shape.
4. Create a Python list containing every column name.
5. Select four clinically meaningful columns.
6. Identify every numeric column.
7. Identify every text column.
8. Count missing values in each column.
9. Calculate the percentage missing in each column.
10. Count exact duplicate rows.
11. Find the number of unique values in every column.
12. Sort by a numeric variable from highest to lowest.
13. Filter records using one numeric condition.
14. Filter records using two conditions joined with `&` or `|`.
15. Create a frequency table for one categorical variable.
16. Create a grouped summary using `groupby()`.
17. Create a cross-tabulation of two categorical variables.
18. Create one derived variable.
19. Create one data-quality review flag.
20. Create a histogram, bar chart, boxplot, or scatterplot and write a two-sentence interpretation.
21. Export your final DataFrame as a CSV file.
22. Write a five-row cleaning log.

In [ ]:
# Practice Laboratory workspace

# Exploration Laboratory – From Raw Data to Analytic Insight

Select a PostgreSQL table that you have not yet analyzed in detail. Without step-by-step instructions, produce a brief exploratory report that answers:

1. What does one row represent?
2. How many rows and columns are present?
3. Which variables are most relevant to a meaningful healthcare question?
4. What missingness, duplication, coding, type, or range issues are present?
5. What cleaning decisions are defensible?
6. Which observations should be flagged for review rather than deleted?
7. What are the most important descriptive findings?
8. What chart best communicates one finding?
9. Is the cleaned dataset fit for the proposed purpose?
10. What limitations should accompany the analysis?

Your submission should include executable code, output, interpretation, and a cleaning log.

In [ ]:
# Exploration Laboratory workspace

# My Professional Perspective

Python became valuable to me because it made analytic work reproducible. In a spreadsheet, it is easy to change a value, move a column, or apply a formula without leaving a complete record of what happened. A well-designed notebook preserves the sequence of questions, code, results, and interpretations.

The most important skill is not memorizing every pandas command. It is learning to inspect the data before acting, make explicit decisions, preserve source values, and communicate the consequences of those decisions. In healthcare, the cleaning process can influence who is included, how risk is characterized, and what leaders believe is happening. Reproducibility is therefore both a technical discipline and a professional responsibility.

# Clinical Decision Point

A quality analyst imports 50,000 blood-pressure records and discovers:

- 7% of systolic values are missing;
- 300 rows are exact duplicates;
- several patients have multiple valid readings;
- 42 systolic values are below 60 or above 260;
- smoking status is recorded as `Yes`, `Y`, `yes`, `No`, `N`, and missing;
- the most recent month contains substantially more missing values than prior months.

Before calculating a hypertension rate, decide:

1. Which records, if any, should be removed?
2. Which records should be flagged rather than deleted?
3. How should smoking status be standardized?
4. Should missing blood pressures be imputed, excluded, or reported separately?
5. Could the recent increase in missingness represent a workflow or data-feed problem?
6. What information must be documented before presenting results?

There may be more than one defensible answer. Your recommendation must be transparent and supported by reasoning.

# Reflection

Write one paragraph responding to each prompt:

1. Which data-quality issue would be most likely to change a clinical or operational conclusion?
2. How did preserving a raw copy change your approach to cleaning?
3. Which cleaning decision required the most judgment?
4. What would you ask the data owner before completing the analysis?

# Knowledge Check

1. What is the difference between a Python variable and a DataFrame column?
2. Why should the raw imported DataFrame be preserved?
3. What does `df.shape` return?
4. What is the difference between `df.isna().sum()` and `df.nunique()`?
5. Why is a duplicate patient identifier not always an error?
6. What does `errors="coerce"` do in a conversion function?
7. Why should an implausible value often be flagged before it is deleted?
8. What is the purpose of `groupby()`?
9. When is a cross-tabulation useful?
10. What does a histogram communicate?
11. Why should chart axes and titles be labeled?
12. What information belongs in a cleaning log?
13. What is the difference between raw data and an analysis-ready dataset?
14. Why are reproducible transformations important in healthcare analytics?
15. What evidence would support a conclusion that a dataset is fit for purpose?

# Stretch Challenge – Build a Reusable Data Profiler

Write a Python function that accepts any DataFrame and returns:

- rows and columns;
- data types;
- missing counts and percentages;
- unique-value counts;
- duplicate-row count;
- numeric minimum, median, mean, and maximum;
- a list of potential identifier columns;
- a list of potential categorical columns.

Use the function on two different database tables and compare the results.

In [ ]:
# Stretch Challenge workspace

def profile_dataframe(dataframe):
    # Build and return a reusable profile.
    pass

# Connection to the Group Project

## Module 4 Deliverable: Analysis-Ready Dataset and Data-Cleaning Notebook

Each team will:

1. Import the cohort or query result produced in Module 3.
2. Preserve an unchanged raw DataFrame.
3. Inspect dimensions, variables, data types, and distributions.
4. Evaluate missingness, duplicates, categories, and valid ranges.
5. Document every cleaning decision.
6. Create at least two analytically useful variables.
7. Produce at least three descriptive summaries.
8. Create at least two exploratory visualizations.
9. Export an analysis-ready dataset for Module 5.
10. Write a fitness-for-purpose statement and limitations paragraph.

The notebook should run from beginning to end using the **Python 3.12.12** kernel.

# Module Summary

In this module, you moved from SQL retrieval to a reproducible Python workflow. You connected to PostgreSQL, imported data into pandas, inspected the DataFrame, evaluated data quality, cleaned selected issues, created analytic variables, summarized the population, produced exploratory charts, documented decisions, and exported an analysis-ready dataset.

The next module will build on this dataset to introduce statistical analysis and inference.